# Phase 2 — Supervised Fine-Tuning (SFT) — 5 Trials

**Base model:** `TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T`  
**Dataset:** `tatsu-lab/alpaca` (10,000 samples subset)  
**Method:** LoRA via PEFT + TRL SFTTrainer + 4-bit NF4 quantization

**Before running:**
1. Mount Google Drive (Cell 2).
2. Upload `test_prompts.json` (with filled reference answers) to the session or Drive.
3. Update `PROMPTS_PATH` and `DRIVE_OUTPUT_DIR` in Cell 4 if needed.
4. Runtime → Change runtime type → **T4 GPU**.

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install -q transformers==4.44.2 peft==0.12.0 trl==0.9.6 \
    bitsandbytes==0.43.3 accelerate==0.34.2 datasets==2.21.0 \
    evaluate==0.4.3 bert-score==0.3.13 sacrebleu==2.4.3

In [ ]:
# ── Cell 2: Mount Google Drive ─────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Cell 3: Imports ────────────────────────────────────────────────────────────
import gc
import json
import os
import time

import evaluate
import pandas as pd
import torch
from bert_score import score as bert_score_fn
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, TaskType
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM

print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ── Cell 4: Config ─────────────────────────────────────────────────────────────
MODEL_ID          = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
PROMPTS_PATH      = "test_prompts.json"         # upload to session or set Drive path
DRIVE_OUTPUT_DIR  = "/content/drive/MyDrive/sft_trials"   # adapters saved here
SFT_RESULTS_PATH  = "/content/drive/MyDrive/sft_results.csv"
DATASET_N_TRAIN   = 9000     # 90% of 10k subset
DATASET_N_VAL     = 1000     # 10% of 10k subset
MAX_SEQ_LENGTH    = 512
MAX_NEW_TOKENS    = 200
TEMPERATURE       = 0.7
TOP_P             = 0.9

os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

In [ ]:
# ── Cell 5: Load and inspect alpaca dataset ────────────────────────────────────
raw = load_dataset("tatsu-lab/alpaca", split="train")
raw = raw.shuffle(seed=42).select(range(10000))
split = raw.train_test_split(test_size=0.1, seed=42)
train_ds = split["train"]
val_ds   = split["test"]
print(f"Train: {len(train_ds)}  |  Val: {len(val_ds)}")
print("Sample:", train_ds[0])

In [ ]:
# ── Cell 6: Dataset formatting ─────────────────────────────────────────────────
RESPONSE_TEMPLATE = "### Response:\n"

def format_alpaca(example):
    if example["input"].strip():
        text = (
            "Below is an instruction that describes a task, paired with an input "
            "that provides further context. Write a response that appropriately "
            "completes the request.\n\n"
            f"### Instruction:\n{example['instruction']}\n\n"
            f"### Input:\n{example['input']}\n\n"
            f"### Response:\n{example['output']}"
        )
    else:
        text = (
            "Below is an instruction that describes a task. Write a response that "
            "appropriately completes the request.\n\n"
            f"### Instruction:\n{example['instruction']}\n\n"
            f"### Response:\n{example['output']}"
        )
    return {"text": text}

train_ds = train_ds.map(format_alpaca, remove_columns=train_ds.column_names)
val_ds   = val_ds.map(format_alpaca,   remove_columns=val_ds.column_names)
print("Formatted. Sample:\n", train_ds[0]["text"][:300])

In [ ]:
# ── Cell 7: Load test prompts for evaluation ───────────────────────────────────
with open(PROMPTS_PATH) as f:
    data = json.load(f)

test_prompts    = [p["prompt"]    for p in data["prompts"]]
test_references = [p["reference"] for p in data["prompts"]]

empty = [i+1 for i, r in enumerate(test_references) if not r.strip()]
if empty:
    raise ValueError(f"Missing references for prompt IDs: {empty}")

sacrebleu = evaluate.load("sacrebleu")
print(f"Loaded {len(test_prompts)} test prompts.")

In [ ]:
# ── Cell 8: Evaluation helper ──────────────────────────────────────────────────
def evaluate_model(model, tokenizer, prompts, references):
    """Run BLEU + BERTScore on 10 test prompts. Returns (mean_bleu, mean_bert_f1, generated_texts)."""
    model.eval()
    generated = []
    for prompt in prompts:
        formatted = (
            "Below is an instruction that describes a task. Write a response that "
            "appropriately completes the request.\n\n"
            f"### Instruction:\n{prompt}\n\n### Response:\n"
        )
        inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
        input_len = inputs["input_ids"].shape[1]
        with torch.inference_mode():
            out = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=True,
                temperature=TEMPERATURE,
                top_p=TOP_P,
                pad_token_id=tokenizer.eos_token_id,
            )
        text = tokenizer.decode(out[0][input_len:], skip_special_tokens=True).strip()
        generated.append(text)

    bleu_scores = [
        sacrebleu.compute(predictions=[g], references=[[r]])["score"]
        for g, r in zip(generated, references)
    ]
    _, _, F1 = bert_score_fn(generated, references, lang="en",
                              rescale_with_baseline=True, verbose=False)
    bert_f1 = [f.item() for f in F1]

    return round(sum(bleu_scores)/len(bleu_scores), 4), \
           round(sum(bert_f1)/len(bert_f1), 4), \
           generated

In [ ]:
# ── Cell 9: 4-bit BnB config (shared across all trials) ───────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

In [ ]:
# ── Cell 10: Trial configurations ─────────────────────────────────────────────
# Each config: lora params + training hyperparams
TRIAL_CONFIGS = [
    {
        "name": "T1",
        "desc": "Minimal LoRA (r=4), 1 epoch",
        "lora": {
            "r": 4,
            "lora_alpha": 8,
            "target_modules": ["q_proj", "v_proj"],
            "lora_dropout": 0.05,
        },
        "train": {
            "learning_rate": 2e-4,
            "per_device_train_batch_size": 4,
            "gradient_accumulation_steps": 4,
            "num_train_epochs": 1,
        },
    },
    {
        "name": "T2",
        "desc": "r=8, same modules, 2 epochs",
        "lora": {
            "r": 8,
            "lora_alpha": 16,
            "target_modules": ["q_proj", "v_proj"],
            "lora_dropout": 0.05,
        },
        "train": {
            "learning_rate": 2e-4,
            "per_device_train_batch_size": 4,
            "gradient_accumulation_steps": 4,
            "num_train_epochs": 2,
        },
    },
    {
        "name": "T3",
        "desc": "r=16, all attn modules, 2 epochs",
        "lora": {
            "r": 16,
            "lora_alpha": 32,
            "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
            "lora_dropout": 0.05,
        },
        "train": {
            "learning_rate": 1e-4,
            "per_device_train_batch_size": 8,
            "gradient_accumulation_steps": 2,
            "num_train_epochs": 2,
        },
    },
    {
        "name": "T4",
        "desc": "r=32, full LoRA (attn+MLP), low LR",
        "lora": {
            "r": 32,
            "lora_alpha": 64,
            "target_modules": [
                "q_proj", "k_proj", "v_proj", "o_proj",
                "gate_proj", "up_proj", "down_proj",
            ],
            "lora_dropout": 0.05,
        },
        "train": {
            "learning_rate": 5e-5,
            "per_device_train_batch_size": 4,
            "gradient_accumulation_steps": 4,
            "num_train_epochs": 2,
        },
    },
    {
        "name": "T5",
        "desc": "r=8, high LR (5e-4), 1 epoch",
        "lora": {
            "r": 8,
            "lora_alpha": 16,
            "target_modules": ["q_proj", "v_proj"],
            "lora_dropout": 0.05,
        },
        "train": {
            "learning_rate": 5e-4,
            "per_device_train_batch_size": 8,
            "gradient_accumulation_steps": 2,
            "num_train_epochs": 1,
        },
    },
]

In [ ]:
# ── Cell 11: Training loop (runs all 5 trials sequentially) ───────────────────
# Each trial: load fresh base model → apply LoRA → train → eval → save adapter
# Estimated time per trial on T4: ~20-40 min depending on config.

results = []

for cfg in TRIAL_CONFIGS:
    trial_name = cfg["name"]
    print(f"\n{'='*70}")
    print(f"  TRIAL {trial_name}: {cfg['desc']}")
    print(f"{'='*70}")

    # ── Load tokenizer + fresh base model ─────────────────────────────────────
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"  # required for SFT causal LM training

    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
    )
    base_model.config.use_cache = False
    base_model.config.pretraining_tp = 1

    # ── LoRA config ────────────────────────────────────────────────────────────
    lora_config = LoraConfig(
        r=cfg["lora"]["r"],
        lora_alpha=cfg["lora"]["lora_alpha"],
        target_modules=cfg["lora"]["target_modules"],
        lora_dropout=cfg["lora"]["lora_dropout"],
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )

    # ── Output dir ────────────────────────────────────────────────────────────
    output_dir = f"/content/sft_{trial_name}"  # local first, copy to Drive after
    drive_dir  = f"{DRIVE_OUTPUT_DIR}/{trial_name}"

    # ── Training arguments ────────────────────────────────────────────────────
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=cfg["train"]["num_train_epochs"],
        per_device_train_batch_size=cfg["train"]["per_device_train_batch_size"],
        gradient_accumulation_steps=cfg["train"]["gradient_accumulation_steps"],
        per_device_eval_batch_size=4,
        learning_rate=cfg["train"]["learning_rate"],
        warmup_ratio=0.03,
        lr_scheduler_type="cosine",
        weight_decay=0.01,
        fp16=True,
        logging_steps=50,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        report_to="none",
        optim="paged_adamw_8bit",
    )

    # ── SFTTrainer ─────────────────────────────────────────────────────────────
    trainer = SFTTrainer(
        model=base_model,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        peft_config=lora_config,
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        tokenizer=tokenizer,
        args=training_args,
        packing=False,
    )

    # ── Train ──────────────────────────────────────────────────────────────────
    t0 = time.time()
    train_result = trainer.train()
    train_time = time.time() - t0

    # ── Val loss ───────────────────────────────────────────────────────────────
    eval_output = trainer.evaluate()
    val_loss = round(eval_output["eval_loss"], 4)

    # ── Save adapter (local then Drive) ───────────────────────────────────────
    trainer.model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    os.makedirs(drive_dir, exist_ok=True)
    !cp -r {output_dir}/* {drive_dir}/
    print(f"Adapter saved to {drive_dir}")

    # ── Evaluate on test prompts ───────────────────────────────────────────────
    print("Evaluating on 10 test prompts...")
    tokenizer.padding_side = "left"  # switch for generation
    mean_bleu, mean_bert_f1, generated_texts = evaluate_model(
        trainer.model, tokenizer, test_prompts, test_references
    )

    results.append({
        "trial": trial_name,
        "desc": cfg["desc"],
        "r": cfg["lora"]["r"],
        "lora_alpha": cfg["lora"]["lora_alpha"],
        "target_modules": ", ".join(cfg["lora"]["target_modules"]),
        "lr": cfg["train"]["learning_rate"],
        "batch_size": cfg["train"]["per_device_train_batch_size"],
        "grad_accum": cfg["train"]["gradient_accumulation_steps"],
        "epochs": cfg["train"]["num_train_epochs"],
        "val_loss": val_loss,
        "mean_bleu": mean_bleu,
        "mean_bert_f1": mean_bert_f1,
        "train_time_min": round(train_time / 60, 1),
        "adapter_path": drive_dir,
        "generated_texts": generated_texts,
    })

    print(f"\n  {trial_name} | val_loss={val_loss} | BLEU={mean_bleu} | BERTScore F1={mean_bert_f1} | time={train_time/60:.1f}min")

    # ── Free GPU memory before next trial ─────────────────────────────────────
    del trainer, base_model
    gc.collect()
    torch.cuda.empty_cache()

print("\nAll 5 trials complete.")

In [ ]:
# ── Cell 12: Results comparison table ─────────────────────────────────────────
cols = ["trial","desc","r","lora_alpha","target_modules",
        "lr","batch_size","epochs","val_loss","mean_bleu","mean_bert_f1","train_time_min"]

df_sft = pd.DataFrame(results)[cols]
pd.set_option("display.max_colwidth", 50)
print(df_sft.to_string(index=False))

# Save to Drive
df_sft.to_csv(SFT_RESULTS_PATH, index=False)
print(f"\nSaved to {SFT_RESULTS_PATH}")

In [ ]:
# ── Cell 13: Select best SFT model ────────────────────────────────────────────
# Ranking: BERTScore F1 (primary) → BLEU (secondary) → val_loss (tiebreaker)
df_ranked = df_sft.sort_values(
    by=["mean_bert_f1", "mean_bleu", "val_loss"],
    ascending=[False, False, True]
).reset_index(drop=True)

best = df_ranked.iloc[0]
print("\n" + "="*60)
print(f"  BEST SFT TRIAL: {best['trial']}")
print(f"  Description:    {best['desc']}")
print(f"  BERTScore F1:   {best['mean_bert_f1']}")
print(f"  BLEU:           {best['mean_bleu']}")
print(f"  Val Loss:       {best['val_loss']}")
print(f"  Adapter path:   {results[df_ranked.index[0]]['adapter_path']}")
print("="*60)

BEST_SFT_ADAPTER = results[df_ranked.index[0]]["adapter_path"]
print(f"\nSave this path for use in 02_dpo_training.ipynb:\n  {BEST_SFT_ADAPTER}")

In [ ]:
# ── Cell 14: Print generated outputs for best trial (for report) ──────────────
best_idx = df_ranked.index[0]
best_generated = results[best_idx]["generated_texts"]

print(f"Generated outputs from best trial ({best['trial']}):")
for i, (prompt, gen, ref) in enumerate(
    zip(test_prompts, best_generated, test_references)
):
    print(f"\n[{i+1}] {prompt[:80]}")
    print(f"  Generated : {gen[:200]}")
    print(f"  Reference : {ref[:200]}")

## Next Steps

1. Copy the **best adapter path** printed in Cell 13 into `02_dpo_training.ipynb` → `BEST_SFT_ADAPTER_PATH`.
2. Copy the SFT results table into **Section 4 (SFT Experiments)** of `report.md`.
3. Note which trial won and briefly explain why (higher BERTScore F1 → more semantically aligned responses).